# Experiment 4A — Physical Value of Expanded Neighborhood

این نوت‌بوک فقط **کنترل‌کننده**ٔ آزمایش است؛ تمام منطق در `experiment4A_physical_value.py` قرار دارد. سه مرحله را به ترتیب اجرا کن. فایل‌های Full Coverage داخل پوشه `data` قرار داده شده‌اند. برای مرحلهٔ Routing، مسیرهای بخش تنظیمات را مطابق کامپیوتر خودت تعیین کن.

**نکته:** `prepare` نیاز به Qiskit ندارد؛ `explore` و `validate` به Qiskit 0.45.x، نسخهٔ اصلاح‌شدهٔ GidNET، فایل Planهای Experiment 2 و پنج QASM اصلی نیاز دارند. خروجی‌ها خودکار ذخیره می‌شوند و اجرای قطع‌شده از checkpoint ادامه پیدا می‌کند.

In [ ]:
import os

PYTHON045 = Path(
    r"C:\Users\ASUS\caqr-qiskit045-env\Scripts\python.exe"
)

In [ ]:
from pathlib import Path
import subprocess, sys

# پوشه‌ای که فایل experiment4A_physical_value.py و data داخلش قرار دارند
EXP4A_DIR = Path.cwd()

# ریشه‌ی پروژهٔ GidNET را تنظیم کن (باید پوشه‌ی gidnet داخل آن باشد)
PROJECT_ROOT = Path(r"C:\Users\ASUS\Documents\Quantum computing\Qiskit\GidNET-Qubit-Reuse-Algorithm")

# مسیر فولدرِ ۵ فایل QASM را متناسب با کامپیوتر خودت تغییر بده
BENCH_DIR = Path(r"C:\Users\ASUS\Documents\Quantum computing\Qiskit\CaQR-main\CaQR-main\benchmarks")

MUTATIONS = EXP4A_DIR / "data" / "adversarial_mutation_cases.csv"
AUDIT = EXP4A_DIR / "data" / "width_and_original_plan_audit.csv"
PLANS = PROJECT_ROOT / "experiment2_results" / "experiment2_plan_features.csv"
SCRIPT = EXP4A_DIR / "experiment4A_physical_value.py"
OUT = EXP4A_DIR / "experiment4a_results"

print('Python:', sys.executable)
print('Script found:', SCRIPT.is_file())
print('Full Coverage found:', MUTATIONS.is_file(), AUDIT.is_file())
print('Original plans found:', PLANS.is_file())
print('QASM directory found:', BENCH_DIR.is_dir())

In [ ]:
run_stage("explore", max_new_routes=1)

In [ ]:
def run_stage(stage, max_new_routes=None):
    cmd = [str(PYTHON045), str(SCRIPT), stage,
           '--mutations', str(MUTATIONS), '--audit', str(AUDIT),
           '--out', str(OUT)]
    if stage in ('explore','validate'):
        if not PLANS.is_file() or not BENCH_DIR.is_dir():
            raise FileNotFoundError('Original Plan CSV یا فولدر QASM پیدا نشد. اول مسیرهای PLANS و BENCH_DIR را در سلول قبل تنظیم کن.')
        cmd += ['--plans', str(PLANS), '--bench-dir', str(BENCH_DIR)]
    if max_new_routes is not None:
        cmd += ['--max-new-routes', str(max_new_routes)]
    print('Running:', ' '.join(map(str, cmd)))
    env = os.environ.copy()
    env["PYTHONPATH"] = (
        str(PROJECT_ROOT)
        + os.pathsep
        + env.get("PYTHONPATH", "")
    )
    result = subprocess.run(
        cmd,
        cwd=str(PROJECT_ROOT),
        env=env,
        text=True
    )
    if result.returncode:
        raise RuntimeError(f'Experiment 4A stage {stage} failed with exit code {result.returncode}')
    return result

## مرحلهٔ ۱ — آماده‌سازی بدون Routing

این مرحله تعداد همسایه‌ها را بررسی می‌کند و مجموعه کاندیداهای متمایز را می‌سازد.

In [ ]:
run_stage('prepare')
import pandas as pd
print(pd.read_csv(OUT / 'coverage_per_source.csv').select_dtypes(include='number').sum())

## مرحلهٔ ۲ — Explore با Seedهای ۴۲ تا ۴۶

این بخش **Routing واقعی** را شروع می‌کند؛ اگر برای اولین بار اجرا می‌کنی، می‌توانی ابتدا خط smoke test را اجرا کنی تا فقط ۵ transpile انجام شود. بعد `run_stage("explore")` را اجرا کن.

نتایج به‌طور خودکار ذخیره می‌شوند.

In [ ]:
# تست کوتاه اختیاری:
# run_stage('explore', max_new_routes=5)

# اجرای کامل:
run_stage('explore')

import pandas as pd
display(pd.read_csv(OUT / 'opportunity_summary.csv'))

In [ ]:
import pandas as pd

route_file = OUT / "routes_train.csv"

if route_file.exists():
    df = pd.read_csv(route_file)
    print("Saved routes:", len(df))
    print(df["status"].value_counts())

    if "error" in df.columns:
        print(df[df["status"] != "ok"]["error"].head(10).to_string(index=False))
else:
    print("Routing has not saved any results yet.")

## مرحلهٔ ۳ — ارزیابی Frozen Winners با Seedهای تازهٔ ۱۰۰ تا ۱۲۹

انتخاب‌ها از مرحلهٔ قبل فریز شده‌اند. این مرحله برای جلوگیری از انتخاب دوباره بر اساس نتیجهٔ تصادفی Routing است.

In [ ]:
run_stage('validate')
import pandas as pd
display(pd.read_csv(OUT / 'fresh_seed_comparison.csv').head(15))

## تفسیر درست نتایج

اگر `strict_beneficiaries` مثبت بود، همسایگی گسترده برای آن معیار و آن Plan اولیه **فرصت جدید** ساخته است. اما این آزمایش نشان نمی‌دهد که یک الگوریتم با بودجهٔ محدود بتواند به آن برسد؛ آن مسئلهٔ آزمایش بعدی (Budgeted Guided Repair) است.

این فایل اعتبار ایستای Check شده را مبنا قرار می‌دهد؛ Semantics اجرایی Measurement/Reset را مستقلاً اثبات نمی‌کند.